# Study 5, part 2: measuring what participants wrote

Each of the 223 participants rated 18 second works on transformativeness, essence change and
creativity, and wrote a justification after every rating: 12,042 justifications in all. This notebook
turns those justifications into numbers. It applies Study 1's eight word lists, the rules, without
changing them, and writes the tables the integration file (`Study5_03_Integration.Rmd`) needs for the
two registered text checks and for the exploratory description.

**The measurement is blind by construction.** The notebook reads only five columns of the long data:
the participant, the source image, the image kind, the modification level and the question, beside the
text. It never reads the source condition, the editing time or the rating, so nothing here can depend
on them, and it has no blind and unblinded modes. The integration file joins the condition and the
rating back on.

No person and no language model labels, codes or judges any response. A rule is a search pattern (a
written list of words and phrases, in any of their ordinary forms) that the computer applies in the
same way to every response. The one use of a pretrained model is the check in section 5, in which a
locally stored sentence-embedding model (all-mpnet-base-v2) turns each response into a list of numbers
so that a grouping by meaning can be compared with the topics.

Everything is computed by the shared module `shared/python/study_text.py`, which holds the measurement
half of the Study 1 notebook. This notebook only builds the blind slice, calls the module, checks the
outputs and writes them.

Run it from this folder with the Python 3.12 kernel:

    jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=python312 Study5_02_Text.ipynb

---

# 0. Setup

The paths start from this notebook's folder. Three environment variables (settings the shell passes to
the program) can name other folders: `STUDY5_DIR` the Study 5 folder, `STUDY1_DIR` the Study 1 folder
whose deposited rules are compared below, and `STUDY_SHARED_PYTHON` the folder of the shared module.
Printed paths are relative to the pipeline folder, so the output names no account and no folder
particular to one machine.

The registration names 20260903 as the seed of the text code. A seed is the number that starts a
random-number generator. The cell below makes it the seed of every random step in this notebook, all
of which sit in the topic model and its agreement check. The seed alone does not make a rerun give the
same numbers. The shared module also runs the k-means step of the agreement check, which splits the
responses into groups of similar meaning, on one processor thread (one strand of computation). On
several threads, k-means adds up its numbers in pieces, one per thread, and combines the pieces in the
order the threads finish, which changes from run to run; since rounding depends on that order, its
groups can differ. The seed and the single thread together make a rerun give the same numbers.

In [1]:
import datetime
import hashlib
import os
import platform
import sys
import warnings

# The Hugging Face libraries that load the embedding model are kept off the
# internet, so nothing is downloaded.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")


def short_warning(message, category, filename, lineno, line=None):
    # A warning names the file it came from without the file's folder.
    return "%s: %s (%s, line %d)\n" % (category.__name__, message, os.path.basename(filename), lineno)


warnings.formatwarning = short_warning

import numpy as np
import pandas as pd

STUDY_DIR = os.path.abspath(os.environ.get("STUDY5_DIR", os.getcwd()))
STUDY1_DIR = os.path.abspath(os.environ.get("STUDY1_DIR", os.path.join(STUDY_DIR, os.pardir, "Study 1")))
SHARED_PYTHON = os.path.abspath(os.environ.get("STUDY_SHARED_PYTHON",
                                               os.path.join(STUDY_DIR, os.pardir, "shared", "python")))
sys.dont_write_bytecode = True
sys.path.insert(0, SHARED_PYTHON)
import study_text

# The registered seed of the text code. The ten refits of the topic model's
# stability rule use 20260903 to 20260912, and the final topic fit and the
# k-means grouping of the agreement check use 20260903.
TEXT_SEED = 20260903
TEXT_SEEDS = study_text.set_text_seed(TEXT_SEED)

INPUT_PATH = os.path.join(STUDY_DIR, "study5_long.csv")
NLP_DIR = os.path.join(STUDY_DIR, "nlp")
TABLES_DIR = os.path.join(STUDY_DIR, "tables")
os.makedirs(NLP_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)
MANIFEST_PATH = os.path.join(TABLES_DIR, "study5_text_run_manifest.csv")
show = study_text.display_path

assert os.path.exists(INPUT_PATH), "Cannot find %s. Run study5_build.py first." % show(INPUT_PATH)

# The previous run's manifest is deleted before anything is written, so a
# manifest on disk means the run that wrote it reached its last cell.
if os.path.exists(MANIFEST_PATH):
    os.remove(MANIFEST_PATH)

RUN_ID = study_text.new_run_id()


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


INPUT_SHA256 = file_sha256(INPUT_PATH)
print("Run:", RUN_ID)
print("Input:", show(INPUT_PATH))
print("Input SHA-256:", INPUT_SHA256)
print("Shared module:", show(study_text.__file__))
print("Seeds:", TEXT_SEEDS)

Run: 20261015T210432Z-c895e1ef
Input: Study 5/study5_long.csv
Input SHA-256: 16ca4802d863c2edecd6e57dbc187a2fd3bd6b70cc26580481619624b3ce38b6
Shared module: shared/python/study_text.py
Seeds: {'stability_refits_first': 20260903, 'stability_refits_last': 20260912, 'final_topic_fit': 20260903, 'kmeans_grouping': 20260903}


---

# 1. The blind slice

The blind slice is the only table the measurement sees. It is read from `study5_long.csv`, which
`study5_build.py` writes, with only the identifier, stimulus, question and text columns loaded, so the
condition and the rating never enter this notebook's memory.

A response is about one second work: a source image at one modification level. The key column
`stimulus_id` therefore names the second work, such as `1Figure_None`, and the source image, the image
kind and the modification level travel beside it as `stimulus_image`, `stimulus_type` and
`stimulus_modification`. The shared module accepts only columns named `stimulus_...` beside the key
and the text, and refuses any other column by name.

In [2]:
READ_COLUMNS = ["id", "stimulus_id", "stimulus_type", "modification", "dimension", "justification"]
raw = pd.read_csv(INPUT_PATH, usecols=READ_COLUMNS, dtype=str, keep_default_na=False)
assert sorted(raw.columns) == sorted(READ_COLUMNS), "Unexpected columns were read."

blind = pd.DataFrame({
    "id": raw["id"],
    "stimulus_id": raw["stimulus_id"] + "_" + raw["modification"],
    "stimulus_image": raw["stimulus_id"],
    "stimulus_type": raw["stimulus_type"],
    "stimulus_modification": raw["modification"],
    "dimension": raw["dimension"],
    "response_text": raw["justification"],
})
del raw
study_text.check_blind(blind)

# The counts the registration states: 12,042 justifications from 223
# participants, none empty, 54 for each participant.
n_rows, n_people = len(blind), blind["id"].nunique()
print("Responses:", n_rows, "| participants:", n_people,
      "| second works:", blind["stimulus_id"].nunique(),
      "| source images:", blind["stimulus_image"].nunique())
print("Questions:", sorted(blind["dimension"].unique()))
assert n_rows == 12042 and n_people == 223
assert blind["stimulus_id"].nunique() == 30 and blind["stimulus_image"].nunique() == 10
assert (blind.groupby("id").size() == 54).all()
assert (blind["response_text"].str.strip() != "").all(), "A justification is empty."
print("Columns:", list(blind.columns))

Responses: 12042 | participants: 223 | second works: 30 | source images: 10
Questions: ['creativity', 'essence_change', 'transformativeness']
Columns: ['id', 'stimulus_id', 'stimulus_image', 'stimulus_type', 'stimulus_modification', 'dimension', 'response_text']


---

# 2. The rules are Study 1's

The registration says Study 1's eight word lists run unchanged. Two checks show that they do.

First, the module's rules, written out as Study 1's notebook wrote them (with Study 1's name and its
2,220 responses), must equal Study 1's deposited `tables/text_rules.txt` character for character, and
their SHA-256 checksum must equal the one in Study 1's `tables/text_rules.sha256`. A checksum is a
64-character string computed from the exact bytes of a text, which changes if one character changes.

Second, Study 5's own rules text may differ from Study 1's in two lines only: the title, which names
the study, and the line that gives the number of responses. The constructed sentences, written for the
check and taken from no corpus, must also be read as the rules say.

In [3]:
STUDY1_RULES = os.path.join(STUDY1_DIR, "tables", "text_rules.txt")
STUDY1_RULES_SHA = os.path.join(STUDY1_DIR, "tables", "text_rules.sha256")
STUDY1_N_RESPONSES = 2220

with open(STUDY1_RULES, encoding="utf-8") as handle:
    study1_rules_text = handle.read()
with open(STUDY1_RULES_SHA, encoding="utf-8") as handle:
    study1_sha = handle.read().split()[0]

as_study1 = study_text.format_rules(study_text.RULES, "Study 1", STUDY1_N_RESPONSES) + "\n"
assert as_study1 == study1_rules_text, "The module's rules differ from Study 1's deposited rules."
assert study_text.sha256_text(as_study1) == study1_sha == study_text.sha256_text(study1_rules_text), \
    "The SHA-256 of the rules differs from Study 1's deposited checksum."
print("Rules equal Study 1's tables/text_rules.txt: True")
print("SHA-256 equals Study 1's tables/text_rules.sha256:", study1_sha)

RULES_TEXT = study_text.rules_text(blind, "Study 5")
RULES_SHA256 = study_text.sha256_text(RULES_TEXT)
differing = [(a, b) for a, b in zip(study1_rules_text.splitlines(), RULES_TEXT.splitlines()) if a != b]
assert len(study1_rules_text.splitlines()) == len(RULES_TEXT.splitlines())
assert len(differing) == 2, "Study 5's rules differ from Study 1's in more than the two named lines."
print()
print("Lines in which Study 5's rules text differs from Study 1's:")
for a, b in differing:
    print("  Study 1:", a)
    print("  Study 5:", b)
print()
print("SHA-256 of Study 5's rules text:", RULES_SHA256)

probes = study_text.check_constructed_sentences()
print("Constructed sentences read as the rules say:", len(probes))

Rules equal Study 1's tables/text_rules.txt: True
SHA-256 equals Study 1's tables/text_rules.sha256: 6aa77dbc26c9086c5349a863c1cab808d7bd3758265fc365de2fb36c46a30239

Lines in which Study 5's rules text differs from Study 1's:
  Study 1: STUDY 1 TEXT MEASURES: THE RULES IN FULL
  Study 5: STUDY 5 TEXT MEASURES: THE RULES IN FULL
  Study 1: Every measure below is applied by computer to all 2,220 responses, identically,
  Study 5: Every measure below is applied by computer to all 12,042 responses, identically,

SHA-256 of Study 5's rules text: 86c9680251b4de72da94c2a57d9e6309141c414d109b480a54ba077998accafe
Constructed sentences read as the rules say: 13


---

# 3. The measures

One row per response: the identifiers, the word count, the eight measures, the reading of machine-made
talk beside its flag, and the four polarity columns that say whether a detail, time, skill or concept
phrase was claimed or denied. The machine-made reading is "asserted" when a response states, plainly
or as an impression, that a machine made the work. The integration file counts those statements in the
Human condition. The counts below are over the whole corpus and are split by nothing.

In [4]:
measures = study_text.measures_frame(blind)
MEASURES_PATH = os.path.join(NLP_DIR, "study5_text_measures.csv")
study_text.write_csv_atomic(MEASURES_PATH, measures, RUN_ID)
print("Written:", show(MEASURES_PATH), "|", len(measures), "rows |", len(measures.columns), "columns")
print("Columns:", list(measures.columns))
print()
print("Responses each flag fires on, whole corpus:")
for flag in study_text.TRUE_FALSE_FLAGS:
    print("  %-20s %6d" % (flag, int(measures[flag].sum())))
print("  %-20s %6d" % ("machine_made_reading asserted",
                       int((measures["machine_made_reading"] == "asserted").sum())))

Written: Study 5/nlp/study5_text_measures.csv | 12042 rows | 20 columns
Columns: ['id', 'stimulus_id', 'stimulus_image', 'stimulus_type', 'stimulus_modification', 'dimension', 'n_words', 'says_detail', 'says_time', 'says_skill', 'says_concept', 'says_prompt', 'says_machine_made', 'machine_made_reading', 'prior_exposure', 'anchor_echo', 'detail_polarity', 'time_polarity', 'skill_polarity', 'concept_polarity']

Responses each flag fires on, whole corpus:
  says_detail             559
  says_time              2149
  says_skill              104
  says_concept           1398
  says_prompt             415
  says_machine_made       195
  anchor_echo             631
  machine_made_reading asserted    164


---

# 4. The word tables for the two registered checks

Three tables carry the words of each response to the integration file. A content word is a word left
after the fixed list of function words, such as *the* and *of*, is removed. The word-count table has one
row per response and content word that appears in at least two responses. The vocabulary lists those
words with the number of responses each appears in, and the content-word table gives each response's
distinct content words. The first check predicts the rating and the source condition from the counts. The second compares how much
participants reuse their own words, using the content-word lists. The registration fixes both checks for
creativity, and the tables cover every question so that the integration file chooses the question.

In [5]:
WORDCOUNTS_PATH = os.path.join(NLP_DIR, "study5_text_wordcounts.csv")
VOCABULARY_PATH = os.path.join(NLP_DIR, "study5_text_vocabulary.csv")
TOKENS_PATH = os.path.join(NLP_DIR, "study5_text_tokens.csv")
wordcounts = study_text.wordcounts_frame(blind)
vocabulary = study_text.vocabulary_frame(blind)
tokens = study_text.tokens_frame(blind)
study_text.write_csv_atomic(WORDCOUNTS_PATH, wordcounts, RUN_ID)
study_text.write_csv_atomic(VOCABULARY_PATH, vocabulary, RUN_ID)
study_text.write_csv_atomic(TOKENS_PATH, tokens, RUN_ID)
for path, frame in [(WORDCOUNTS_PATH, wordcounts), (VOCABULARY_PATH, vocabulary), (TOKENS_PATH, tokens)]:
    print("Written:", show(path), "|", len(frame), "rows")

Written: Study 5/nlp/study5_text_wordcounts.csv | 66040 rows
Written: Study 5/nlp/study5_text_vocabulary.csv | 2159 rows
Written: Study 5/nlp/study5_text_tokens.csv | 12042 rows


---

# 5. The exploratory description

**Tone and concreteness.** Each response gets the mean valence (how pleasant a word is), arousal (how
excited it makes a reader feel), dominance (how much in control it makes a reader feel) and concreteness
of its content words, from published word ratings (Warriner, Kuperman & Brysbaert, 2013; Brysbaert,
Warriner & Kuperman, 2014). The words of the rating questions are removed first. A response with no
matched word gets an empty score.

**Topics.** A topic model finds groups of words that tend to occur in the same responses. The model is
non-negative matrix factorisation, fitted to each question separately and given no labels, with the
number of topics chosen by the stability rule in the module.

**The agreement check.** The sentence-embedding model turns each response into 768 numbers, the
responses to each question are split into as many groups as that question has topics, and the split is
compared with each response's main topic by the adjusted Rand index (agreement between two groupings
beyond chance: 0 for chance, 1 for identical). The ten refits of the stability rule use the seeds
20260903 to 20260912, and the final topic fit and the grouping use 20260903. Nothing the embedding
model produces is kept beyond that index.

In [6]:
AFFECT_PATH = os.path.join(NLP_DIR, "study5_text_affect.csv")
affect = study_text.affect_frame(blind)
study_text.write_csv_atomic(AFFECT_PATH, affect, RUN_ID)
print("Written:", show(AFFECT_PATH), "|", len(affect), "rows")
print("Responses with no word in the valence norms:", int((affect["n_vad_matched"] == 0).sum()))
print("Responses with no word in the concreteness norms:", int((affect["n_conc_matched"] == 0).sum()))

Written: Study 5/nlp/study5_text_affect.csv | 12042 rows
Responses with no word in the valence norms: 1119
Responses with no word in the concreteness norms: 907


In [7]:
TOPICS_PATH = os.path.join(NLP_DIR, "study5_text_topics.csv")
TOP_WORDS_PATH = os.path.join(TABLES_DIR, "study5_text_topics_top_words.csv")
STABILITY_PATH = os.path.join(TABLES_DIR, "study5_text_topics_stability.csv")
AGREEMENT_PATH = os.path.join(TABLES_DIR, "study5_text_topics_embedding_agreement.csv")

topics = study_text.topics_frame(blind)
top_words = study_text.topic_top_words_frame(blind)
stability = study_text.topic_stability_frame(blind)
fit = study_text.fit_topics(blind)
study_text.write_csv_atomic(TOPICS_PATH, topics, RUN_ID)
study_text.write_csv_atomic(TOP_WORDS_PATH, top_words, RUN_ID)
study_text.write_csv_atomic(STABILITY_PATH, stability, RUN_ID)
for dim, fitted in fit["by_dimension"].items():
    print("%-18s topics: %d | vocabulary: %d | refits stopped at the iteration limit: %d of %d | final fit settled: %s"
          % (dim, fitted["k"], fitted["n_vocab"], fitted["n_short"], fitted["n_refits"],
             fitted["final_converged"]))
print()
print(top_words[top_words["rank"] <= 5].groupby(["dimension", "topic"])["word"]
      .apply(", ".join).to_string())

agreement = study_text.embedding_agreement_frame(blind)
EMBEDDING_STATUS = agreement.attrs.get("embedding_status", "")
study_text.write_csv_atomic(AGREEMENT_PATH, agreement, RUN_ID)
print()
print("Embedding model:", study_text.EMBEDDING_MODEL_NAME, "|", EMBEDDING_STATUS)
print(agreement.to_string(index=False))

creativity         topics: 6 | vocabulary: 548 | refits stopped at the iteration limit: 0 of 80 | final fit settled: True
essence_change     topics: 8 | vocabulary: 537 | refits stopped at the iteration limit: 0 of 80 | final fit settled: True
transformativeness topics: 7 | vocabulary: 527 | refits stopped at the iteration limit: 0 of 80 | final fit settled: True

dimension           topic
creativity          1                  first, second, different, new, style
                    2                  change, color, style, little, slight
                    3               changed, color, colors, style, lighting
                    4                   look, identical, exact, ai, exactly
                    5                  difference, see, tell, notice, color
                    6                   looks, exact, ai, generated, filter
essence_change      1              different, style, feel, vibe, completely
                    2                    second, first, unique, felt, feels


Embedding model: sentence-transformers/all-mpnet-base-v2 | loaded from the local copy
         dimension  k    ari
        creativity  6 0.0735
    essence_change  8 0.1269
transformativeness  7 0.0817


---

# 6. Export

The rules, their checksum and their grounding notes go to `tables/`, with a record of the software and a
run manifest. Every data file is read back from disk and held to its agreed columns, none of which is
the condition, the rating or the text. The manifest lists every output with its size and SHA-256,
beside the run identifier, the input's SHA-256 and the rules checksum, and it is written last.

In [8]:
RULES_PATH = os.path.join(TABLES_DIR, "study5_text_rules.txt")
RULES_SHA_PATH = os.path.join(TABLES_DIR, "study5_text_rules.sha256")
GROUNDING_PATH = os.path.join(TABLES_DIR, "study5_text_rules_grounding.txt")
ENVIRONMENT_PATH = os.path.join(TABLES_DIR, "study5_text_environment.csv")

study_text.write_text_atomic(RULES_PATH, RULES_TEXT, RUN_ID)
# The checksum line in the format `shasum -a 256 -c` reads, from the tables folder.
study_text.write_text_atomic(RULES_SHA_PATH, "%s  study5_text_rules.txt\n" % RULES_SHA256, RUN_ID)
study_text.write_text_atomic(GROUNDING_PATH, study_text.grounding_text(blind, "Study 5"), RUN_ID)
assert file_sha256(RULES_PATH) == RULES_SHA256


def package_version(module_name):
    try:
        module = __import__(module_name)
    except Exception:
        return "not installed"
    return str(getattr(module, "__version__", "installed, version not recorded"))


environment = pd.DataFrame([
    ("run_id", RUN_ID),
    ("python", platform.python_version()),
    ("platform", platform.platform()),
    ("pandas", pd.__version__),
    ("numpy", np.__version__),
    ("scikit-learn", package_version("sklearn")),
    ("scipy", package_version("scipy")),
    ("sentence-transformers", package_version("sentence_transformers")),
    ("torch", package_version("torch")),
    ("embedding_model", "%s, %s" % (study_text.EMBEDDING_MODEL_NAME, EMBEDDING_STATUS)),
    ("seed_text_code", str(TEXT_SEED)),
    ("seed_stability_refits", "%d to %d" % (TEXT_SEEDS["stability_refits_first"],
                                            TEXT_SEEDS["stability_refits_last"])),
    ("seed_final_topic_fit", str(TEXT_SEEDS["final_topic_fit"])),
    ("seed_kmeans_grouping", str(TEXT_SEEDS["kmeans_grouping"])),
    ("kmeans_openmp_threads", str(study_text.KMEANS_OPENMP_THREADS)),
], columns=["item", "value"])
# The seeds in force at the end of the run are the ones set at its start.
assert study_text.text_seeds() == TEXT_SEEDS
study_text.write_csv_atomic(ENVIRONMENT_PATH, environment, RUN_ID)
print(environment.to_string(index=False))
print()

# Every data file, read back from disk and held to its columns. The files
# keyed by response, apart from the word counts (which have no row for a
# response with no counted word), must cover exactly the 12,042 responses.
FORBIDDEN = ["Type", "source", "editing_time", "deriv_effort", "rating", "response_text",
             "condition", "arm", "age", "gender", "experience"]
CONTRACTS = [
    (MEASURES_PATH, study_text.export_columns(blind)),
    (WORDCOUNTS_PATH, study_text.WORDCOUNT_COLUMNS),
    (VOCABULARY_PATH, study_text.VOCABULARY_COLUMNS),
    (TOKENS_PATH, study_text.TOKENS_COLUMNS),
    (AFFECT_PATH, study_text.AFFECT_COLUMNS),
    (TOPICS_PATH, study_text.TOPICS_COLUMNS),
]
keys = set(blind[study_text.KEY].itertuples(index=False, name=None))
for path, columns in CONTRACTS:
    on_disk = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert list(on_disk.columns) == list(columns), "%s: columns differ." % show(path)
    assert not [c for c in FORBIDDEN if c in on_disk.columns], "%s carries a forbidden column." % show(path)
    if "dimension" in columns and path != WORDCOUNTS_PATH:
        assert set(on_disk[study_text.KEY].itertuples(index=False, name=None)) == keys, \
            "%s does not cover exactly the 12,042 responses." % show(path)
    print("  %-38s %7d rows, columns as agreed, no forbidden column" % (show(path), len(on_disk)))

outputs = [MEASURES_PATH, WORDCOUNTS_PATH, VOCABULARY_PATH, TOKENS_PATH, AFFECT_PATH, TOPICS_PATH,
           TOP_WORDS_PATH, STABILITY_PATH, AGREEMENT_PATH, RULES_PATH, RULES_SHA_PATH,
           GROUNDING_PATH, ENVIRONMENT_PATH]
completed_utc = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
manifest = pd.DataFrame([dict(
    run_id=RUN_ID, completed_utc=completed_utc,
    input_file=os.path.basename(INPUT_PATH), input_sha256=INPUT_SHA256, rules_sha256=RULES_SHA256,
    file=os.path.relpath(path, STUDY_DIR), bytes=os.path.getsize(path), sha256=file_sha256(path),
) for path in outputs])
study_text.write_csv_atomic(MANIFEST_PATH, manifest, RUN_ID)
print()
print("Written last:", show(MANIFEST_PATH), "|", len(manifest), "outputs listed")

                 item                                                               value
               run_id                                           20261015T210432Z-c895e1ef
               python                                                              3.12.3
             platform                                        macOS-26.6.2-arm64-arm-64bit
               pandas                                                               2.2.2
                numpy                                                              1.26.4
         scikit-learn                                                               1.5.2
                scipy                                                              1.13.0
sentence-transformers                                                               3.3.1
                torch                                                               2.5.1
      embedding_model sentence-transformers/all-mpnet-base-v2, loaded from the local copy
       see